## 1. Imports and configuration

In [ ]:
import os
import json
from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# Paths
INPUT_FOLDER  = Path("DatasetsCSV") / "train"
OUTPUT_FOLDER = Path("DatasetsProcessed")

# Temporal parameters
RESOLUTION_MIN = 15      # common grid resolution in minutes
WINDOW_LENGTH  = 96      # 96 bins * 15 min = 24 h
STRIDE         = 12      # 12 bins * 15 min = 3 h

# CGM gap handling
MAX_INTERP_BINS = 2      # interpolate gaps up to 2 consecutive bins (30 min)
MIN_SEGMENT_LEN = WINDOW_LENGTH

# Physiological limits for normalisation
PHYS_LIMITS = {
    "cgm":   (40.0, 400.0),    # mg/dL
    "basal": (0.0,  3.0),      # U/h
    "bolus": (0.0,  25.0),     # U
    "carbs": (0.0,  200.0),    # g (upper clip to handle outliers)
}

CHANNELS = ["cgm", "basal", "bolus", "carbs"]
CHANNEL_UNITS = {"cgm": "mg/dL", "basal": "U/h", "bolus": "U", "carbs": "g"}


## 2. Load patient CSVs

In [ ]:
def discover_patients(folder: Path) -> List[str]:
    """Returns sorted list of patient IDs found in the input folder."""
    ids = []
    for fname in os.listdir(folder):
        if fname.startswith("glucose_") and fname.endswith(".csv"):
            pid = fname.replace("glucose_", "").replace(".csv", "")
            ids.append(pid)
    return sorted(ids)


def load_patient_csvs(pid: str, folder: Path) -> Dict[str, pd.DataFrame]:
    """Loads the four signal CSVs for a patient. Returns empty DataFrame if file is missing."""
    sources = ["glucose", "basal", "bolus", "meal"]
    out = {}
    for source in sources:
        path = folder / f"{source}_{pid}.csv"
        if not path.exists():
            out[source] = pd.DataFrame()
            continue
        df = pd.read_csv(path)
        if "ts" in df.columns:
            df["ts"] = pd.to_datetime(df["ts"], errors="coerce")
            df = df.dropna(subset=["ts"]).sort_values("ts").reset_index(drop=True)
        out[source] = df
    return out


## 3. Build multivariate series on a 15-minute grid

Each signal is resampled according to its nature:
- **CGM**: mean of native 5-min readings per bin. Empty bins → NaN.
- **Basal**: forward-filled between changes, then averaged per bin. Bins before the first record → 0.
- **Bolus**: sum per bin. Empty bins → 0.
- **Carbs**: sum per bin with upper clip at 200 g. Empty bins → 0.


In [ ]:
def build_grid(t_min: pd.Timestamp, t_max: pd.Timestamp, freq_min: int) -> pd.DatetimeIndex:
    """Builds an evenly-spaced grid covering [t_min, t_max]."""
    freq = f"{freq_min}min"
    return pd.date_range(start=t_min.floor(freq), end=t_max.ceil(freq), freq=freq)


def resample_cgm(df_glucose: pd.DataFrame, grid: pd.DatetimeIndex, freq_min: int) -> pd.Series:
    """Resamples CGM to the grid by mean. Empty bins → NaN."""
    if df_glucose.empty:
        return pd.Series(np.nan, index=grid, name="cgm")
    s = (df_glucose.set_index("ts")["value"]
                   .astype(float)
                   .resample(f"{freq_min}min")
                   .mean())
    return s.reindex(grid).rename("cgm")


def resample_basal(df_basal: pd.DataFrame, grid: pd.DatetimeIndex, freq_min: int) -> pd.Series:
    """Resamples basal rate: forward-fill between changes, then mean per bin."""
    if df_basal.empty:
        return pd.Series(0.0, index=grid, name="basal")
    s = df_basal.set_index("ts")["value"].astype(float)
    minute_grid = pd.date_range(start=grid[0], end=grid[-1], freq="1min")
    s_minute = s.reindex(minute_grid, method="ffill")
    # Bins before the first record are unknown — set to 0
    s_minute = s_minute.fillna(0.0)
    s_grid = s_minute.resample(f"{freq_min}min").mean()
    return s_grid.reindex(grid).rename("basal")


def resample_bolus(df_bolus: pd.DataFrame, grid: pd.DatetimeIndex, freq_min: int) -> pd.Series:
    """Resamples bolus: sum of doses per bin. Empty bins → 0."""
    if df_bolus.empty:
        return pd.Series(0.0, index=grid, name="bolus")
    s = (df_bolus.set_index("ts")["dose"]
                 .astype(float)
                 .resample(f"{freq_min}min")
                 .sum())
    return s.reindex(grid).fillna(0.0).rename("bolus")


def resample_carbs(df_meal: pd.DataFrame, grid: pd.DatetimeIndex,
                   freq_min: int, clip_max: float) -> pd.Series:
    """Resamples carbs: sum per bin with upper clip. Empty bins → 0."""
    if df_meal.empty:
        return pd.Series(0.0, index=grid, name="carbs")
    s = (df_meal.set_index("ts")["carbs"]
                .astype(float)
                .resample(f"{freq_min}min")
                .sum())
    return s.reindex(grid).fillna(0.0).clip(upper=clip_max).rename("carbs")


def build_patient_series(data: Dict[str, pd.DataFrame],
                         freq_min: int, carbs_clip: float) -> pd.DataFrame:
    """Combines the four signals into a multivariate DataFrame on the 15-min grid."""
    ts_mins, ts_maxs = [], []
    for source, df in data.items():
        if not df.empty and "ts" in df.columns:
            ts_mins.append(df["ts"].min())
            ts_maxs.append(df["ts"].max())
    if not ts_mins:
        raise ValueError("No temporal data found in any source.")
    grid = build_grid(min(ts_mins), max(ts_maxs), freq_min)

    cgm   = resample_cgm  (data["glucose"], grid, freq_min)
    basal = resample_basal(data["basal"],   grid, freq_min)
    bolus = resample_bolus(data["bolus"],   grid, freq_min)
    carbs = resample_carbs(data["meal"],    grid, freq_min, carbs_clip)

    df = pd.concat([cgm, basal, bolus, carbs], axis=1)
    df.index.name = "ts"
    return df


## 4. CGM gap segmentation

Short NaN gaps (≤ MAX_INTERP_BINS) are filled by linear interpolation.
Longer gaps are used as cut points, splitting the series into contiguous segments.
Segments shorter than one window length are discarded.


In [ ]:
def interpolate_short_gaps(df: pd.DataFrame, col: str, max_gap: int) -> Tuple[pd.DataFrame, int]:
    """Linearly interpolates NaN runs of length <= max_gap in column col."""
    df = df.copy()
    s = df[col]
    is_na = s.isna()
    if not is_na.any():
        return df, 0

    run_id = (is_na != is_na.shift()).cumsum()
    run_lengths = is_na.groupby(run_id).transform("sum")
    interpolatable = is_na & (run_lengths <= max_gap)

    s_interp = s.interpolate(method="linear", limit=max_gap, limit_direction="both")
    new_s = s.copy()
    new_s[interpolatable] = s_interp[interpolatable]

    df[col] = new_s
    return df, int(interpolatable.sum())


def split_into_segments(df: pd.DataFrame, gap_col: str, min_len: int) -> List[pd.DataFrame]:
    """Splits the DataFrame into contiguous NaN-free segments. Discards segments shorter than min_len."""
    is_valid = df[gap_col].notna()
    block_id = (is_valid != is_valid.shift()).cumsum()

    segments = []
    for _, group in df.groupby(block_id):
        if group[gap_col].notna().all() and len(group) >= min_len:
            segments.append(group.copy())
    return segments


## 5. Min-Max normalisation

Each channel is normalised to [-1, 1] using fixed physiological limits.
Values outside the declared range are clipped as a defensive measure.


In [ ]:
def normalise_segment(seg: pd.DataFrame, limits: Dict[str, Tuple[float, float]]) -> pd.DataFrame:
    """Applies Min-Max normalisation to [-1, 1] using fixed physiological limits per channel."""
    out = seg.copy()
    for channel, (lo, hi) in limits.items():
        if channel not in out.columns:
            continue
        x = out[channel].astype(float).values
        x_norm = 2.0 * (x - lo) / (hi - lo) - 1.0
        out[channel] = np.clip(x_norm, -1.0, 1.0)
    return out


## 6. Sliding window extraction

Extracts windows of length L=96 (24 h) with stride S=12 (3 h).
Channel order: [cgm, basal, bolus, carbs].


In [ ]:
def windows_from_segment(seg: pd.DataFrame, channels: List[str],
                          L: int, stride: int) -> np.ndarray:
    """Extracts sliding windows from a single segment. Returns array of shape (n_windows, L, C)."""
    arr = seg[channels].to_numpy(dtype=np.float32)
    T, C = arr.shape
    if T < L:
        return np.empty((0, L, C), dtype=np.float32)
    starts = np.arange(0, T - L + 1, stride)
    return np.stack([arr[s: s + L] for s in starts], axis=0)


def windows_from_patient(segments: List[pd.DataFrame], channels: List[str],
                          L: int, stride: int) -> np.ndarray:
    """Concatenates windows from all segments of a patient."""
    if not segments:
        return np.empty((0, L, len(channels)), dtype=np.float32)
    blocks = [windows_from_segment(seg, channels, L, stride) for seg in segments]
    return np.concatenate(blocks, axis=0)


## 7. Patient pipeline

Runs the full preprocessing pipeline for a single patient and saves windows.npy and metadata.json.


In [ ]:
def raw_stats(df: pd.DataFrame) -> Dict[str, Dict[str, float]]:
    """Descriptive statistics per channel before normalisation."""
    out = {}
    for channel in df.columns:
        s = df[channel].dropna()
        if channel in ("bolus", "carbs"):
            out[channel] = {
                "min": float(s.min()) if len(s) else None,
                "max": float(s.max()) if len(s) else None,
                "mean": float(s.mean()) if len(s) else None,
                "n_nonzero": int((s != 0).sum()),
            }
        else:
            out[channel] = {
                "min": float(s.min()) if len(s) else None,
                "max": float(s.max()) if len(s) else None,
                "mean": float(s.mean()) if len(s) else None,
            }
    return out


def process_patient(pid: str, input_folder: Path,
                    output_folder: Path, verbose: bool = True) -> Dict:
    """Full preprocessing pipeline for one patient. Saves windows.npy and metadata.json."""
    data = load_patient_csvs(pid, input_folder)

    df = build_patient_series(data, freq_min=RESOLUTION_MIN,
                              carbs_clip=PHYS_LIMITS["carbs"][1])
    stats = raw_stats(df)
    n_total_bins = len(df)
    n_cgm_nan = int(df["cgm"].isna().sum())

    df, n_imputed = interpolate_short_gaps(df, "cgm", MAX_INTERP_BINS)
    stats["cgm"]["n_imputed"] = n_imputed

    segments = split_into_segments(df, "cgm", MIN_SEGMENT_LEN)
    seg_lens = [len(s) for s in segments]

    segments_norm = [normalise_segment(s, PHYS_LIMITS) for s in segments]
    windows = windows_from_patient(segments_norm, CHANNELS, WINDOW_LENGTH, STRIDE)

    coverage = float(sum(seg_lens) / n_total_bins) if n_total_bins > 0 else 0.0

    out_dir = output_folder / f"patient_{pid}"
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / "windows.npy", windows)

    metadata = {
        "patient_id": pid,
        "channels": CHANNELS,
        "channel_units": [CHANNEL_UNITS[c] for c in CHANNELS],
        "window_length": WINDOW_LENGTH,
        "stride": STRIDE,
        "resolution_min": RESOLUTION_MIN,
        "n_windows": int(windows.shape[0]),
        "n_segments": len(segments),
        "segment_lengths": seg_lens,
        "n_total_bins": n_total_bins,
        "n_cgm_nan_inicial": n_cgm_nan,
        "coverage_after_segmentation": coverage,
        "physiological_limits": {k: list(v) for k, v in PHYS_LIMITS.items()},
        "preprocessing": {
            "interpolation_max_bins": MAX_INTERP_BINS,
            "carbs_clip_max": PHYS_LIMITS["carbs"][1],
            "exercise_included": False,
        },
        "raw_stats_pre_norm": stats,
        "time_range": {
            "start": str(df.index.min()),
            "end":   str(df.index.max()),
        },
    }
    with open(out_dir / "metadata.json", "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, ensure_ascii=False)

    if verbose:
        print(f"Patient {pid} processed.")

    return metadata


## 8. Process all patients

In [ ]:
def process_all(input_folder: Path = INPUT_FOLDER,
                output_folder: Path = OUTPUT_FOLDER) -> List[Dict]:
    """Runs the pipeline for all patients found in the input folder."""
    output_folder.mkdir(parents=True, exist_ok=True)
    patients = discover_patients(input_folder)

    metadata_list = []
    for pid in patients:
        meta = process_patient(pid, input_folder, output_folder, verbose=True)
        metadata_list.append(meta)

    print("Complete.")
    return metadata_list


metadata_list = process_all()


## 9. Excel export for manual inspection

Generates a single .xlsx file with one sheet per patient, in long format and original scale (mg/dL, U/h, U, g).

In [ ]:
def denormalise(x_norm: np.ndarray, lo: float, hi: float) -> np.ndarray:
    """Inverts Min-Max normalisation from [-1, 1] back to original scale."""
    return (x_norm + 1.0) / 2.0 * (hi - lo) + lo


def windows_to_long_df(windows: np.ndarray, channels: List[str],
                        limits: Dict[str, Tuple[float, float]],
                        resolution_min: int) -> pd.DataFrame:
    """Converts a (N, L, C) normalised tensor to a long-format DataFrame in original scale."""
    if windows.shape[0] == 0:
        return pd.DataFrame(columns=["window_id", "step", "timestamp_offset_h"] + channels)

    N, L, C = windows.shape
    windows_orig = np.empty_like(windows, dtype=np.float64)
    for c_idx, channel in enumerate(channels):
        lo, hi = limits[channel]
        windows_orig[:, :, c_idx] = denormalise(windows[:, :, c_idx].astype(np.float64), lo, hi)

    window_ids = np.repeat(np.arange(N), L)
    steps = np.tile(np.arange(L), N)
    offsets_h = steps * (resolution_min / 60.0)

    df = pd.DataFrame({"window_id": window_ids, "step": steps, "timestamp_offset_h": offsets_h})
    flat = windows_orig.reshape(N * L, C)
    for c_idx, channel in enumerate(channels):
        df[channel] = flat[:, c_idx]
    return df


def export_excel(metadata_list: List[Dict], output_folder: Path = OUTPUT_FOLDER,
                 filename: str = "dataset_complete.xlsx") -> Path:
    """Writes a single .xlsx with one sheet per patient."""
    out_path = output_folder / filename
    with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
        for m in metadata_list:
            pid = m["patient_id"]
            pdir = output_folder / f"patient_{pid}"
            windows = np.load(pdir / "windows.npy")
            limits = {k: tuple(v) for k, v in m["physiological_limits"].items()}
            df_long = windows_to_long_df(windows, m["channels"], limits, m["resolution_min"])
            df_long.to_excel(writer, sheet_name=f"patient_{pid}", index=False)
            print(f"Patient {pid} processed.")
    print("Complete.")
    return out_path


export_excel(metadata_list)


## 10. Visual inspection

In [ ]:
def load_processed_patient(pid: str, output_folder: Path = OUTPUT_FOLDER):
    """Loads windows.npy and metadata.json for a processed patient."""
    pdir = output_folder / f"patient_{pid}"
    windows = np.load(pdir / "windows.npy")
    with open(pdir / "metadata.json", "r", encoding="utf-8") as f:
        meta = json.load(f)
    return windows, meta


def empirical_acf(x: np.ndarray, max_lag: int) -> np.ndarray:
    """Computes the empirical autocorrelation function up to max_lag."""
    x = x - x.mean()
    var = np.dot(x, x) / len(x)
    if var == 0:
        return np.zeros(max_lag + 1)
    return np.array([
        np.dot(x[: len(x) - lag], x[lag:]) / (len(x) * var)
        for lag in range(max_lag + 1)
    ])


def plot_patient(pid: str, output_folder: Path = OUTPUT_FOLDER):
    """Three diagnostic plots per patient: CGM histogram, example window, ACF."""
    windows, meta = load_processed_patient(pid, output_folder)
    if windows.shape[0] == 0:
        print(f"Patient {pid}: no windows available.")
        return

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    fig.suptitle(f"Patient {pid} — {windows.shape[0]} windows", fontsize=12)

    # CGM histogram (normalised)
    cgm_flat = windows[:, :, 0].flatten()
    axes[0].hist(cgm_flat, bins=60, color="steelblue", alpha=0.85)
    axes[0].set_title("CGM (normalised [-1, 1])")
    axes[0].set_xlabel("normalised value")
    axes[0].set_ylabel("frequency")
    axes[0].axvline(-1, color="red", linewidth=0.5, linestyle="--")
    axes[0].axvline( 1, color="red", linewidth=0.5, linestyle="--")

    # Example window
    idx = windows.shape[0] // 2
    t_axis = np.arange(WINDOW_LENGTH) * RESOLUTION_MIN / 60.0
    for c, name in enumerate(CHANNELS):
        axes[1].plot(t_axis, windows[idx, :, c], label=name, linewidth=1.0)
    axes[1].set_title(f"Example window (idx={idx})")
    axes[1].set_xlabel("hours")
    axes[1].set_ylabel("normalised value")
    axes[1].legend(fontsize=8, loc="upper right")
    axes[1].set_ylim(-1.1, 1.1)

    # ACF
    r = empirical_acf(windows[:, :, 0].flatten(), max_lag=48)
    axes[2].stem(np.arange(len(r)) * RESOLUTION_MIN / 60.0, r, basefmt=" ")
    axes[2].set_title("CGM empirical ACF (12 h)")
    axes[2].set_xlabel("lag (hours)")
    axes[2].set_ylabel("r")
    axes[2].axhline(0, color="black", linewidth=0.5)

    plt.tight_layout()
    plt.show()


for m in metadata_list:
    plot_patient(m["patient_id"])


## 11. Sanity checks

In [ ]:
def sanity_checks(metadata_list: List[Dict], output_folder: Path = OUTPUT_FOLDER) -> None:
    """Runs automatic checks on the preprocessed dataset: shape, dtype, range, NaN/Inf."""
    issues = 0
    for m in metadata_list:
        pid = m["patient_id"]
        windows, _ = load_processed_patient(pid, output_folder)

        if windows.shape[0] == 0:
            print(f"  Patient {pid}: 0 windows")
            issues += 1
            continue
        if windows.shape[1:] != (WINDOW_LENGTH, len(CHANNELS)):
            print(f"  Patient {pid}: unexpected shape {windows.shape}")
            issues += 1
        if windows.dtype != np.float32:
            print(f"  Patient {pid}: dtype {windows.dtype} (expected float32)")
            issues += 1
        if windows.min() < -1.0 - 1e-6 or windows.max() > 1.0 + 1e-6:
            print(f"  Patient {pid}: values outside [-1, 1]")
            issues += 1
        if not np.isfinite(windows).all():
            print(f"  Patient {pid}: contains NaN or Inf")
            issues += 1

    if issues == 0:
        total_windows = sum(m['n_windows'] for m in metadata_list)
        print(f"All checks passed ({len(metadata_list)} patients, {total_windows} windows).")
    else:
        print(f"{issues} issue(s) detected.")


sanity_checks(metadata_list)


## 12. Summary table

In [ ]:
rows = []
for pdir in sorted(OUTPUT_FOLDER.glob("patient_*")):
    pid = pdir.name.replace("patient_", "")
    meta_path = pdir / "metadata.json"
    if not meta_path.exists():
        continue
    with open(meta_path) as f:
        m = json.load(f)
    tr = m["time_range"]
    duration_days = (pd.Timestamp(tr["end"]) - pd.Timestamp(tr["start"])).total_seconds() / 86400
    rows.append({
        "patient": pid,
        "duration_days": round(duration_days, 1),
        "n_total_bins": m["n_total_bins"],
        "cgm_nan": m["n_cgm_nan_inicial"],
        "pct_nan": round(m["n_cgm_nan_inicial"] / m["n_total_bins"] * 100, 1),
        "n_segments": m["n_segments"],
        "coverage_pct": round(m["coverage_after_segmentation"] * 100, 1),
        "n_windows": m["n_windows"],
    })

summary = pd.DataFrame(rows).sort_values("patient").reset_index(drop=True)
print(summary.to_string(index=False))
